
You are given NGS genomic data, sequenced from E.Coli strain with knockout
of one gene. Your goal is to determine that gene and describe it.
To achieve this we will use annotated reference genome. Workflow:
1. Download reference genome and reads
2. Check quality of reads
3. Align reads on reference
4. Download and prepare reference annotation
5. Map reads alignment to annotation, find missing gene
6. Search gene in database and interpret results

In [6]:
%%bash
pwd
ls -ahl data

/home/aalarkin/ITMO_CS_MSc/Intro_to_Bioinf/Prak1
total 20M
drwxrwxr-x 2 aalarkin aalarkin 4.0K Sep  8 14:30 .
drwxrwxr-x 5 aalarkin aalarkin 4.0K Sep  8 14:43 ..
-rw-rw-r-- 1 aalarkin aalarkin 2.4M Aug 29 16:45 genomic.gff
-rw-rw-r-- 1 aalarkin aalarkin  13M Aug 29 17:54 reads.fastq.gz
-rw-rw-r-- 1 aalarkin aalarkin 4.5M Aug 30 01:21 sequence.fasta


In [4]:
%%bash

fastqc data/reads.fastq.gz -o results/

application/gzip


Started analysis of reads.fastq.gz
Approx 5% complete for reads.fastq.gz
Approx 10% complete for reads.fastq.gz
Approx 15% complete for reads.fastq.gz
Approx 20% complete for reads.fastq.gz
Approx 25% complete for reads.fastq.gz
Approx 30% complete for reads.fastq.gz
Approx 35% complete for reads.fastq.gz
Approx 40% complete for reads.fastq.gz
Approx 45% complete for reads.fastq.gz
Approx 50% complete for reads.fastq.gz
Approx 55% complete for reads.fastq.gz
Approx 60% complete for reads.fastq.gz
Approx 65% complete for reads.fastq.gz
Approx 70% complete for reads.fastq.gz
Approx 75% complete for reads.fastq.gz
Approx 80% complete for reads.fastq.gz
Approx 85% complete for reads.fastq.gz
Approx 90% complete for reads.fastq.gz
Approx 95% complete for reads.fastq.gz
Approx 100% complete for reads.fastq.gz


Analysis complete for reads.fastq.gz


In [5]:
%%bash

bwa index data/sequence.fasta

[bwa_index] Pack FASTA... 0.04 sec
[bwa_index] Construct BWT for the packed sequence...
[bwa_index] 1.41 seconds elapse.
[bwa_index] Update BWT... 0.02 sec
[bwa_index] Pack forward-only FASTA... 0.02 sec
[bwa_index] Construct SA from BWT and Occ... 0.24 sec
[main] Version: 0.7.19-r1273
[main] CMD: bwa index data/sequence.fasta
[main] Real time: 2.425 sec; CPU: 1.741 sec


In [6]:
%%bash
bwa mem -t 4 data/sequence.fasta data/reads.fastq.gz 2> results/reads.bwa.log > results/reads.sam

In [7]:
%%bash

samtools view -b -t 4 results/reads.sam > results/reads.bam

#Sort reads in the order of occurrence in reference,
samtools sort -t 4 results/reads.bam > results/reads.sorted.bam

#Index alignment file for fast processing (required for IGV, GATK, ...)
samtools index results/reads.sorted.bam

In [9]:
%%bash
samtools faidx data/sequence.fasta

In [10]:
%%bash

grep -v "^#" data/genomic.gff | awk '{FS="\t";OFS="\t"} $3 ~ "CDS"' > data/genomic.filtered.gff
    
awk '{FS="\t";OFS="\t"} {print $1,$4-1,$5,$3,$9}' data/genomic.filtered.gff > data/genomic.filtered.bed

sort -k 1 -nk 2 data/genomic.filtered.bed > results/genomic.filtered.sorted.bed

In [11]:
%%bash

bedtools bamtobed -i results/reads.sorted.bam | cut -f 1-3 > results/reads.sorted.bed
bedtools intersect -c -a results/genomic.filtered.sorted.bed -b results/reads.sorted.bed > results/intersect.bed

awk '{FS="\t";OFS="\t"} $6==0' results/intersect.bed > results/missing_genes.bed

In [14]:
%%bash 

less results/missing_genes.bed

NC_000913.3	103981	105244	CDS	ID=cds-NP_414636.1;Parent=gene-b0094;Dbxref=UniProtKB/Swiss-Prot:P0ABH0,GenBank:NP_414636.1,ASAP:ABE-0000331,ECOCYC:EG10339,GeneID:944778;Name=NP_414636.1;gbkey=CDS;gene=ftsA;locus_tag=b0094;product=cell division protein FtsA;protein_id=NP_414636.1;transl_table=11	0


In [ ]:
#https://docs.google.com/forms/d/e/1FAIpQLSdqHiBkMmkhJ5--l5-kIzMjb4Bn-MAsl8Jsyxgqjivqr2fBdg/viewform?usp=send_form

### HW problems

Count number of reads in fastq file

1. Calculate the length of the read in fastq file

2. Calculate the GC-content in % of all reads in fastq file

3. Write the full name of the reference sequence in fasta file

4. Calculate the length of the reference sequence in fasta file

5. Calculate the number of regions annotated as "mobile_genetic_element" in gff file.

6. Calculate the number of regions annotated as "rRNA" in gff file.

7. Find the start position of "origin_of_replication" in gff file.

8. Find the name of knocked-out gene in data. 

9. Enter the value of "gene" field in 9th column of gff for region not covered with reads

10. Find the role of knocked-out gene in databases and literature.
Write 5-10 sentences describing the function of the encoded protein, its role in organism, possible effects of its knockout. Add links to relevant studies and databases.


In [33]:
%%bash
chmod +r data/reads.fastq.gz
nplusone=$(zcat data/reads.fastq.gz | head -n 2 | tail -n 1 | wc -c)
n=$((nplusone - 1))
echo $n

126


In [40]:
%%bash
nGC=$(zcat data/reads.fastq.gz | awk 'NR%4==2' | grep -o -E "G|C" | wc -l)
nATGC=$(zcat data/reads.fastq.gz | awk 'NR%4==2' | grep -o -E "A|T|G|C" | wc -l)
gcpercent=$(echo "scale=2; ($nGC / $nATGC) * 100" | bc)
echo $gcpercent


50.00


In [63]:
%%bash

#grep "^>" data/sequence.fasta

sp=$(cat data/sequence.fasta | head -n 1 | grep -o "Escherichia.*MG1655")
echo $sp

Escherichia coli str. K-12 substr. MG1655


In [97]:
%%bash
nsymb=$(cat data/sequence.fasta | awk 'NR!=1'| wc -c)
nrows=$(cat data/sequence.fasta | awk 'NR!=1'| wc -l)
nsymb=$(( $nsymb - $nrows ))
echo $nsymb

4641652


In [103]:
cat data/genomic.gff | awk '{FS="\t";OFS="\t"} $3 ~ "mobile_genetic_element"' | wc -l

50


In [104]:
cat data/genomic.gff | awk '{FS="\t";OFS="\t"} $3 ~ "rRNA"' | wc -l

22


In [114]:
cat data/genomic.gff | awk '{FS="\t";OFS="\t"} $3 ~ "origin_of_replication" {print $4}' 

3925744


In [116]:
cat data/genomic.gff | awk '{FS="\t";OFS="\t"} $0 ~ "NP_414636.1"' 

NC_000913.3	RefSeq	CDS	103982	105244	.	+	0	ID=cds-NP_414636.1;Parent=gene-b0094;Dbxref=UniProtKB/Swiss-Prot:P0ABH0,GenBank:NP_414636.1,ASAP:ABE-0000331,ECOCYC:EG10339,GeneID:944778;Name=NP_414636.1;gbkey=CDS;gene=ftsA;locus_tag=b0094;product=cell division protein FtsA;protein_id=NP_414636.1;transl_table=11
